# Cuaderno de Entrenamiento Intensivo — Solemnes Taller de Programación II

**Institución:** Universidad San Sebastián · Sede Patagonia  
**Asignatura:** Taller de Programación II (II° Semestre 2026)  
**Evaluaciones Objetivo:**  
*  **Control 1 (Laboratorio):** Jueves 3 de Septiembre (17:00 hrs · Sala A306 · Ponderación 10%)  
*  **Solemne 1 (Prueba Escrita):** Jueves 10 de Septiembre (17:00 hrs · Sala A306 · Ponderación 20%)  

---

## 1. Pilares Teóricos y Resumen Sintáctico de I/O

### 1.1 Tabla Comparativa de Modos de Apertura (`open()`)

| Modo | Propósito | Puntero Inicial | Crea si no existe | Destruye contenido previo |
| :---: | :--- | :---: | :---: | :---: |
| `'r'` | Lectura pura | Inicio (byte 0) |  Lanza `FileNotFoundError` |  No |
| `'w'` | Escritura desde cero | Inicio (byte 0) |  Sí |  **Sí (Truncamiento a 0 bytes)** |
| `'a'` | Anexión al final | Final del archivo |  Sí |  No |
| `'r+'`| Lectura y escritura in-place | Inicio (byte 0) |  Lanza `FileNotFoundError` |  No |

### 1.2 Regla de Oro del Módulo `csv`
En archivos CSV se **debe** utilizar siempre `newline=""` en la función `open()` para que la capa de E/S de Python no traduzca los saltos de línea produciendo **filas vacías duplicadas** (`\r\r\n`) en Windows:
```python
with open("datos.csv", "w", newline="", encoding="utf-8") as f:
    escritor = csv.writer(f, delimiter=";")
```

### 1.3 Matriz de Métodos JSON

| Método | Tipo de Entrada / Salida | Canal de Trabajo | Ejemplo |
| :--- | :--- | :--- | :--- |
| `json.dump(obj, f)` | Objeto Python $\rightarrow$ Archivo | Disco físico | `json.dump(datos, f, indent=4)` |
| `json.load(f)` | Archivo $\rightarrow$ Objeto Python | Disco físico | `datos = json.load(f)` |
| `json.dumps(obj)` | Objeto Python $\rightarrow$ String `str` | Cadena en memoria | `s = json.dumps(datos)` |
| `json.loads(s)` | String `str` $\rightarrow$ Objeto Python | Cadena en memoria | `datos = json.loads(s)` |

## 2. Las 12 Trampas Frecuentes en Pruebas Escritas (Code Tracing)

> [WARNING]
> En la Solemne 1 escrita se evalúa tu capacidad para rastrear código en papel (*code tracing*) y detectar fallos silenciosos o excepciones en tiempo de ejecución.

1. **Trampa 1 (Truncamiento accidental):** Abrir en modo `'w'` cuando se requería anexar datos en modo `'a'`.
2. **Trampa 2 (Salto de línea pegado):** Escribir registros con `f.write(texto)` sin `\n`, provocando que queden unificados en una sola línea horizontal.
3. **Trampa 3 (Saltos dobles en CSV):** Omitir `newline=""` en `open("archivo.csv", "w")`.
4. **Trampa 4 (Encabezado como entero):** Iterar sobre `csv.reader` sin invocar `next(lector)`, causando `ValueError: invalid literal for int() avec 'Cantidad'`.
5. **Trampa 5 (Split frágil):** Aplicar `split(";")` sobre campos con texto entrecomillado que contienen el delimitador.
6. **Trampa 6 (Confusión dump vs dumps):** Intentar hacer `json.dump(obj)` pasando una cadena en lugar del descriptor de archivo.
7. **Trampa 7 (Lectura sin verificación):** Abrir en modo `'r'` un archivo no existente sin envolver en `try-except FileNotFoundError`.
8. **Trampa 8 (Basura de fin de línea):** Olvidar `.strip()` al leer líneas, manteniendo el `\n` residual en comparaciones.
9. **Trampa 9 (Falsa modificación en disco):** Modificar la lista en memoria RAM pero olvidar escribir el resultado en el archivo.
10. **Trampa 10 (Sensibilidad a mayúsculas):** Comparar cadenas `"Juan"` == `"juan"` arrojando `False` por omitir `.lower()`.
11. **Trampa 11 (Descriptores abiertos):** Abrir sin el bloque `with open(...)` dejando archivos bloqueados en el Kernel.
12. **Trampa 12 (JSON vacío):** Intentar ejecutar `json.load(f)` sobre un archivo de 0 bytes arrojando `json.JSONDecodeError`.

## 3. Ejercicios Prácticos Ejecutables en Celda

### Ejercicio 1. Persistencia TXT y Lectura Defensiva con Try-Except

In [ ]:
from pathlib import Path

RUTA_TXT = Path("datos_entrenamiento.txt")

def guardar_registro(linea: str) -> None:
    # Anexa una línea con salto de línea explícito.
    with RUTA_TXT.open("a", encoding="utf-8") as f:
        f.write(f"{linea.strip()}\n")

def leer_registros_seguro() -> list[str]:
    # Lee el archivo capturando FileNotFoundError.
    try:
        with RUTA_TXT.open("r", encoding="utf-8") as f:
            return [l.strip() for l in f if l.strip()]
    except FileNotFoundError:
        print(" Diagnóstico: Archivo no encontrado. Se retorna lista vacía.")
        return []

guardar_registro("Registro 1: Control de I/O")
guardar_registro("Registro 2: Persistencia Segura")
print("Contenido leído:", leer_registros_seguro())

Contenido leído: ['Registro 1: Control de I/O', 'Registro 2: Persistencia Segura']


### Ejercicio 2. Manejo Estructurado CSV con Encabezado Único y next()

In [ ]:
import csv

RUTA_CSV = Path("alumnos_taller.csv")

def inicializar_csv():
    if not RUTA_CSV.exists():
        with RUTA_CSV.open("w", newline="", encoding="utf-8") as f:
            escritor = csv.writer(f, delimiter=";")
            escritor.writerow(["RUT", "Nombre", "Nota_Control1"])

def agregar_alumno(rut: str, nombre: str, nota: float):
    inicializar_csv()
    with RUTA_CSV.open("a", newline="", encoding="utf-8") as f:
        escritor = csv.writer(f, delimiter=";")
        escritor.writerow([rut.strip(), nombre.strip(), nota])

def leer_alumnos():
    if not RUTA_CSV.exists():
        return []
    with RUTA_CSV.open("r", newline="", encoding="utf-8") as f:
        lector = csv.reader(f, delimiter=";")
        encabezado = next(lector) # Consumir cabecera
        return list(lector)

agregar_alumno("20123456-7", "Moisés", 7.0)
print("Filas leídas:", leer_alumnos())

Filas leídas: [['20123456-7', 'Moisés', '7.0']]


### Ejercicio 3. Serialización JSON y Manejo de JSONDecodeError

In [ ]:
import json

RUTA_JSON = Path("estado_estudiante.json")

datos_estudiante = {
    "alumno": "Moisés",
    "ramo": "Taller de Programación II",
    "meta_nota": 7.0,
    "unidades_completadas": ["TXT", "CSV", "JSON"]
}

with RUTA_JSON.open("w", encoding="utf-8") as f:
    json.dump(datos_estudiante, f, indent=4, ensure_ascii=False)

with RUTA_JSON.open("r", encoding="utf-8") as f:
    recuperado = json.load(f)

print("Objeto JSON recuperado:")
print(f"Alumno: {recuperado['alumno']} | Meta: {recuperado['meta_nota']}")

Objeto JSON recuperado:
Alumno: Moisés | Meta: 7.0
